# 6.5 — Automatic differentiation & computational graphs

Autodiff records the computation graph, then applies exact chain-rule bookkeeping through every node.

Reverse-mode autodiff is backprop as bookkeeping over a graph. We build a tiny graph, verify adjoints, and compare against finite differences. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(6)


def clf_digits_ladder():
    """D1 XOR -> D2 blobs -> D3 noisy moons -> D4 digits -> D5 noisy digits."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 XOR", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=1.0, random_state=1)
    rungs.append(("D2 blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.3, random_state=2)
    rungs.append(("D3 noisy moons", x3, y3))

    digits = load_digits()
    xd = digits.data / 16.0
    rungs.append(("D4 digits (real, 10-class, 64-D)", xd, digits.target))

    rng = np.random.default_rng(5)
    xn = xd + rng.normal(0.0, 0.25, size=xd.shape)
    yn = digits.target.copy()
    flip = rng.random(yn.shape) < 0.1
    yn[flip] = rng.integers(0, 10, size=int(flip.sum()))
    rungs.append(("D5 digits + label/feature noise", xn, yn))

    return rungs


def split_scale(X, y):
    if len(y) < 20:
        scaler = StandardScaler()
        x_scaled = scaler.fit_transform(X)
        return x_scaled, x_scaled, y.copy(), y.copy(), scaler

    x_train, x_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.4,
        random_state=0,
        stratify=y,
    )
    rng = np.random.default_rng(606)
    if len(y_train) > 600:
        train_idx = rng.choice(len(y_train), size=600, replace=False)
        y_train = y_train[train_idx]
        x_train = x_train[train_idx]
    if len(y_test) > 300:
        test_idx = rng.choice(len(y_test), size=300, replace=False)
        y_test = y_test[test_idx]
        x_test = x_test[test_idx]

    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train)
    x_test = scaler.transform(x_test)
    return x_train, x_test, y_train, y_test, scaler


def one_hot(y, classes):
    out = np.zeros((len(y), classes))
    out[np.arange(len(y)), y.astype(int)] = 1.0
    return out


def stable_softmax(logits):
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum(axis=1, keepdims=True)


def activation_forward(z, name):
    if name == "relu":
        return np.maximum(0.0, z)
    if name == "tanh":
        return np.tanh(z)
    if name == "sigmoid":
        return 1.0 / (1.0 + np.exp(-np.clip(z, -40.0, 40.0)))
    if name == "gelu":
        c = math.sqrt(2.0 / math.pi)
        return 0.5 * z * (1.0 + np.tanh(c * (z + 0.044715 * z**3)))
    return z


def activation_backward(z, name):
    if name == "relu":
        return (z > 0.0).astype(float)
    if name == "tanh":
        h = np.tanh(z)
        return 1.0 - h**2
    if name == "sigmoid":
        h = 1.0 / (1.0 + np.exp(-np.clip(z, -40.0, 40.0)))
        return h * (1.0 - h)
    if name == "gelu":
        c = math.sqrt(2.0 / math.pi)
        u = c * (z + 0.044715 * z**3)
        t = np.tanh(u)
        sech2 = 1.0 - t**2
        return 0.5 * (1.0 + t) + 0.5 * z * sech2 * c * (1.0 + 3.0 * 0.044715 * z**2)
    return np.ones_like(z)


def initialize_mlp(n_features, n_hidden, n_classes, seed):
    rng = np.random.default_rng(seed)
    w1 = rng.normal(0.0, math.sqrt(2.0 / max(1, n_features)), size=(n_features, n_hidden))
    b1 = np.zeros(n_hidden)
    w2 = rng.normal(0.0, math.sqrt(2.0 / max(1, n_hidden)), size=(n_hidden, n_classes))
    b2 = np.zeros(n_classes)
    return {"w1": w1, "b1": b1, "w2": w2, "b2": b2}


def forward(params, X, activation):
    z1 = X @ params["w1"] + params["b1"]
    h1 = activation_forward(z1, activation)
    logits = h1 @ params["w2"] + params["b2"]
    probs = stable_softmax(logits)
    cache = {"z1": z1, "h1": h1, "logits": logits, "probs": probs}
    return probs, cache


def compute_loss(probs, y, loss_name):
    classes = probs.shape[1]
    targets = one_hot(y, classes)
    eps = 1e-9

    if loss_name == "mse":
        return float(np.mean((probs - targets) ** 2))

    if loss_name == "hinge":
        correct = probs[np.arange(len(y)), y]
        margins = np.maximum(0.0, probs - correct[:, None] + 0.2)
        margins[np.arange(len(y)), y] = 0.0
        return float(np.mean(np.sum(margins, axis=1)))

    return float(-np.mean(np.log(probs[np.arange(len(y)), y] + eps)))


def output_gradient(probs, y, loss_name):
    classes = probs.shape[1]
    targets = one_hot(y, classes)
    n = max(1, len(y))

    if loss_name == "mse":
        return 2.0 * (probs - targets) / (n * classes)

    if loss_name == "hinge":
        correct = probs[np.arange(len(y)), y]
        active = probs - correct[:, None] + 0.2 > 0.0
        active[np.arange(len(y)), y] = False
        grad = active.astype(float)
        grad[np.arange(len(y)), y] = -grad.sum(axis=1)
        return grad / n

    return (probs - targets) / n


def train_tiny_mlp(
    X,
    y,
    hidden=24,
    epochs=25,
    lr=0.12,
    activation="relu",
    loss_name="ce",
    seed=0,
):
    x_train, x_test, y_train, y_test, scaler = split_scale(X, y)
    classes = int(np.max(y)) + 1
    params = initialize_mlp(x_train.shape[1], hidden, classes, seed)
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(epochs):
        probs, cache = forward(params, x_train, activation)
        grad_logits = output_gradient(probs, y_train, loss_name)
        grad_w2 = cache["h1"].T @ grad_logits
        grad_b2 = grad_logits.sum(axis=0)
        grad_h = grad_logits @ params["w2"].T
        grad_z1 = grad_h * activation_backward(cache["z1"], activation)
        grad_w1 = x_train.T @ grad_z1
        grad_b1 = grad_z1.sum(axis=0)

        params["w1"] = params["w1"] - lr * grad_w1
        params["b1"] = params["b1"] - lr * grad_b1
        params["w2"] = params["w2"] - lr * grad_w2
        params["b2"] = params["b2"] - lr * grad_b2

        train_probs, _ = forward(params, x_train, activation)
        val_probs, _ = forward(params, x_test, activation)
        train_pred = train_probs.argmax(axis=1)
        val_pred = val_probs.argmax(axis=1)
        history["train_loss"].append(compute_loss(train_probs, y_train, loss_name))
        history["val_loss"].append(compute_loss(val_probs, y_test, loss_name))
        history["train_acc"].append(float(accuracy_score(y_train, train_pred)))
        history["val_acc"].append(float(accuracy_score(y_test, val_pred)))

    result = {
        "params": params,
        "history": history,
        "scaler": scaler,
        "x_test": x_test,
        "y_test": y_test,
        "activation": activation,
        "loss_name": loss_name,
    }
    return result


def predict_tiny(model, X_raw):
    X = model["scaler"].transform(X_raw)
    probs, _ = forward(model["params"], X, model["activation"])
    return probs.argmax(axis=1)


def evaluate_ladder(hidden=24, epochs=25, lr=0.12, activation="relu", loss_name="ce", seed=0):
    rows = []
    models = []
    for idx, (name, X, y) in enumerate(clf_digits_ladder(), start=1):
        model = train_tiny_mlp(
            X,
            y,
            hidden=hidden,
            epochs=epochs,
            lr=lr,
            activation=activation,
            loss_name=loss_name,
            seed=seed + idx,
        )
        metric = model["history"]["val_acc"][-1]
        loss_value = model["history"]["val_loss"][-1]
        rows.append({"rung": idx, "name": name, "accuracy": metric, "loss": loss_value})
        models.append(model)
    return rows, models


def print_rows(rows, metric="accuracy"):
    print("rung | dataset | accuracy | loss")
    for row in rows:
        print(f"D{row['rung']} | {row['name']} | {row['accuracy']:.3f} | {row['loss']:.3f}")


def plot_decision_panel(ax, model, X, y, title):
    if X.shape[1] != 2:
        side = int(math.sqrt(X.shape[1]))
        if side * side == X.shape[1]:
            ax.imshow(X[0].reshape(side, side), cmap="gray")
            pred = predict_tiny(model, X[:1])[0]
            ax.set_title(f"{title}\ny={int(y[0])}, pred={int(pred)}")
        else:
            ax.plot(X[0])
            ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])
        return

    x_min = X[:, 0].min() - 0.8
    x_max = X[:, 0].max() + 0.8
    y_min = X[:, 1].min() - 0.8
    y_max = X[:, 1].max() + 0.8
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 40), np.linspace(y_min, y_max, 40))
    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = predict_tiny(model, grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=0.25, levels=np.arange(int(np.max(y)) + 2) - 0.5)
    ax.scatter(X[:, 0], X[:, 1], c=y, s=16, edgecolor="k", linewidth=0.2)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])


def closing_figure(rows, models, metric="accuracy"):
    rungs = clf_digits_ladder()
    fig, axes = plt.subplots(2, 5, figsize=(16, 6))
    for ax, model, rung in zip(axes[0], models, rungs):
        name, X, y = rung
        plot_decision_panel(ax, model, X, y, name.split(" (")[0])

    xs = [row["rung"] for row in rows]
    ys = [row[metric] for row in rows]
    axes[1, 0].plot(xs, ys, marker="o")
    axes[1, 0].set_xticks(xs)
    axes[1, 0].set_xlabel("ladder rung")
    axes[1, 0].set_ylabel(metric)
    axes[1, 0].set_title(f"{metric} vs ladder difficulty")
    for extra in axes[1, 1:]:
        extra.axis("off")
    fig.tight_layout()
    plt.show()


def plot_history(models, metric="val_acc"):
    plt.figure(figsize=(7, 4))
    for idx, model in enumerate(models, start=1):
        plt.plot(model["history"][metric], label=f"D{idx}")
    plt.xlabel("epoch")
    plt.ylabel(metric)
    plt.legend(ncol=3)
    plt.title(f"Training trace: {metric}")
    plt.show()


## Build the concept once on D1

The lesson formula is

$$\bar v=\sum_{u\in children(v)}\bar u\,\frac{\partial u}{\partial v}$$

We plug in the lesson's own numbers and assert the exact rounded values before using the method on the ladder.

In [ ]:

class Value:
    def __init__(self, data, children=(), op=""):
        self.data = float(data)
        self.grad = 0.0
        self.prev = list(children)
        self.op = op
        self._backward = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def backward():
            self.grad += out.grad
            other.grad += out.grad

        out._backward = backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = backward
        return out

    def relu(self):
        out = Value(max(0.0, self.data), (self,), "relu")

        def backward():
            self.grad += (1.0 if self.data > 0.0 else 0.0) * out.grad

        out._backward = backward
        return out

    def backward(self):
        topo = []
        seen = set()

        def build(node):
            if id(node) in seen:
                return
            seen.add(id(node))
            for child in node.prev:
                build(child)
            topo.append(node)

        build(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()


def autograd_graph():
    x0 = Value(1.5)
    x1 = Value(-0.5)
    w0 = Value(1.0)
    w1 = Value(-0.4)
    b = Value(0.8)
    z = w0 * x0 + w1 * x1 + b
    h = z.relu()
    h.backward()
    return z, h, w0, w1, b

z, h, w0, w1, b = autograd_graph()
updated = 2.0 - 0.050 * 1.950
prob = math.exp(z.data) / (math.exp(z.data) + math.exp(0.4))
normalized = (z.data - 1.0) / math.sqrt(0.250 + 0.00001)
memory_kb = 3 * 128 * 4 / 1024

assert round(z.data, 3) == 2.500
assert round(h.data, 3) == 2.500
assert round(w0.grad, 3) == 1.500
assert round(w1.grad, 3) == -0.500
assert round(updated, 3) == 1.903
assert round(math.exp(z.data), 3) == 12.182
assert round(prob, 3) == 0.891
assert round(normalized, 3) == 3.000
assert round(memory_kb, 3) == 1.500

print("node z/h and adjoints dw0,dw1,db:", z.data, h.data, w0.grad, w1.grad, b.grad)


## Package the reusable method

The same tiny MLP code above performs a real forward pass, computes a real loss, backpropagates analytic gradients, and updates weights on CPU.

In [ ]:
rungs = clf_digits_ladder()
print("Reusable method: train_tiny_mlp + forward + analytic backward pass")
print("Rungs ready:", len(rungs))

## The dataset ladder

D1 is XOR, then blobs, noisy moons, real sklearn digits, and D5 digits with feature plus label noise. The method sees one feature matrix and one class vector at every rung.

In [ ]:
for idx, (name, X, y) in enumerate(clf_digits_ladder(), start=1):
    labels, counts = np.unique(y, return_counts=True)
    print(f"D{idx}: {name}")
    print("  X shape:", X.shape)
    print("  classes:", labels.tolist())
    print("  first counts:", counts[:5].tolist())
    print("  first row sample:", np.round(X[0, : min(8, X.shape[1])], 3))

## Run the same method across D1–D5

The tracked metric is loss.

In [ ]:
rows, models = evaluate_ladder(hidden=28, epochs=25, lr=0.12, activation="relu", loss_name="ce", seed=65)
print_rows(rows, metric="loss")

## Results visualization

The closing figure has small-multiple output artifacts for every rung plus the metric curve from D1 to D5. The second plot shows train/validation history so local steps can be compared with generalization.

In [ ]:
closing_figure(rows, models, metric="loss")
plot_history(models, metric="val_loss")

## Pitfall on the hardest rung

D5 is real digits with added feature and label noise, so the pitfall has to show up in a genuinely harder training run rather than a toy picture.

In [ ]:

name, X, y = clf_digits_ladder()[-1]
model = train_tiny_mlp(X, y, hidden=20, epochs=20, lr=0.12, activation="relu", loss_name="ce", seed=655)
params = model["params"]
x_train, x_test, y_train, y_test, scaler = split_scale(X, y)
base_loss = compute_loss(forward(params, x_train[:64], "relu")[0], y_train[:64], "ce")
epsilon = 1e-4
old_value = params["w1"][0, 0]
params["w1"][0, 0] = old_value + epsilon
plus_loss = compute_loss(forward(params, x_train[:64], "relu")[0], y_train[:64], "ce")
params["w1"][0, 0] = old_value - epsilon
minus_loss = compute_loss(forward(params, x_train[:64], "relu")[0], y_train[:64], "ce")
params["w1"][0, 0] = old_value
finite_difference = (plus_loss - minus_loss) / (2.0 * epsilon)
print("graph tensors:", params["w1"].shape, params["b1"].shape, params["w2"].shape)
print("finite-difference check for w1[0,0]:", round(finite_difference, 6))
print("base mini-batch loss:", round(base_loss, 3))


## Evaluate it + Practice

- Metric: inspect D5 loss and compare with a no-skill baseline near the majority-class rate.
- Sanity check: D1 XOR should be learnable by a hidden-layer MLP but not by one linear gate.
- Ablation: reduce width, saturate activations, or use the wrong loss and watch the metric degrade.
- Failure signals: diverging loss, flat validation curves, unstable softmax sums, or gradients with unexpected shapes.

Practice prompts:
1. Change hidden width and replot the D1–D5 curve.

2. Replace ReLU with tanh or GELU and compare D5.

3. Lower the D5 label-noise rate in `clf_digits_ladder()` and predict how the curve changes.